# retrying

## Example 1: retry until the call succeeds

In [ ]:
# Example 1: a function that fails twice is called three times and its third result is returned
from retrying import retry

calls = []

@retry(stop_max_attempt_number=5)
def flaky():
    calls.append(len(calls) + 1)
    if len(calls) < 3:
        raise IOError("not yet")
    return "done"

assert flaky() == "done" and calls == [1, 2, 3], calls
print("Example 1:", flaky.__name__, "succeeded on attempt", calls[-1])

## Example 2: retry while the result is not good enough

In [ ]:
# Example 2: retry_on_result retries while the predicate is true (here: while the result is None)
from retrying import retry

values = iter([None, None, 42])

@retry(retry_on_result=lambda result: result is None, stop_max_attempt_number=5)
def poll():
    return next(values)

assert poll() == 42
print("Example 2: poll() returned 42 after two None results")

## Example 3: retry only the exceptions you choose

In [ ]:
# Example 3: retry_on_exception decides per exception; a ValueError is raised at once, an IOError is retried
from retrying import retry

attempts = {"io": 0, "value": 0}

@retry(retry_on_exception=lambda error: isinstance(error, IOError), stop_max_attempt_number=3)
def fail_with(kind):
    attempts[kind] += 1
    raise IOError("io") if kind == "io" else ValueError("value")

raised = {}
for kind in ("io", "value"):
    try:
        fail_with(kind)
    except Exception as error:
        raised[kind] = type(error).__name__
assert attempts == {"io": 3, "value": 1} and raised == {"io": "OSError", "value": "ValueError"}, (attempts, raised)
print("Example 3: attempts per kind ->", attempts, "| raised ->", raised)

## Example 4: wait between attempts

In [ ]:
# Example 4: wait_fixed sleeps a fixed number of milliseconds between attempts
import time
from retrying import RetryError, retry

stamps = []

@retry(wait_fixed=50, stop_max_attempt_number=3, retry_on_result=lambda r: True)
def always_retry():
    stamps.append(time.monotonic())

try:
    always_retry()
except RetryError as error:
    print("Example 4: stopped with", type(error).__name__, "after attempt", error.last_attempt.attempt_number)
else:
    raise AssertionError("stop_max_attempt_number=3 did not stop the retries")
gaps = [b - a for a, b in zip(stamps, stamps[1:])]
assert len(stamps) == 3 and all(gap >= 0.045 for gap in gaps), gaps
print("Example 4: gaps between attempts (s):", [round(g, 3) for g in gaps])

## Example 5: the RetryError and the last attempt it carries

In [ ]:
# Example 5: when the attempts run out on a bad result, Retrying.call raises RetryError holding the last Attempt
from retrying import RetryError, Retrying

retrying = Retrying(stop_max_attempt_number=4, retry_on_result=lambda r: r < 10)
counter = iter(range(100))
try:
    retrying.call(lambda: next(counter))
except RetryError as error:
    last = error.last_attempt
    assert last.attempt_number == 4 and last.value == 3 and not last.has_exception
    print("Example 5:", error)
else:
    raise AssertionError("RetryError was not raised")